<a href="https://colab.research.google.com/github/chiragjotwani/Adaptive_Ensemble_Learning_for_Smart_Irrigation/blob/main/ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# BLOCK 1: Imports
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

try:
    from xgboost import XGBClassifier
except ImportError:
    import sys
    !{sys.executable} -m pip install -q xgboost
    from xgboost import XGBClassifier


In [ ]:
# BLOCK 2: Data Loading
if "df" not in globals():
    csv_candidates = sorted(Path(".").glob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError("No dataframe named 'df' found and no CSV file is available in the working directory.")
    df = pd.read_csv(csv_candidates[0])

df = df.copy()
df.columns = df.columns.str.strip()
display(df.head())
print(f"Dataset shape: {df.shape}")


In [ ]:
# BLOCK 3: Data Preprocessing
feature_columns = ["temperature", "soil_moisture", "soil_ph", "water_tds", "humidity"]
target_column = "label"

missing_columns = [col for col in feature_columns + [target_column] if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data = df[feature_columns + [target_column]].copy()
data[target_column] = data[target_column].astype(str).str.strip()
data = data.dropna(subset=[target_column]).reset_index(drop=True)

class_counts = data[target_column].value_counts()
rare_classes = class_counts[class_counts < 2].index.tolist()
if rare_classes:
    data.loc[data[target_column].isin(rare_classes), target_column] = "Other"
    if (data[target_column] == "Other").sum() < 2:
        data = data[data[target_column] != "Other"].copy()

class_counts = data[target_column].value_counts()
valid_labels = class_counts[class_counts >= 2].index
data = data[data[target_column].isin(valid_labels)].reset_index(drop=True)

if data[target_column].nunique() < 2:
    raise ValueError("At least two valid classes with >= 2 samples are required after rare-class handling.")

label_encoder = LabelEncoder()
data["label_encoded"] = label_encoder.fit_transform(data[target_column])

X = data[feature_columns].copy()
y = data["label_encoded"].copy()

stratify_target = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target
)

numeric_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]
            ),
            feature_columns,
        )
    ],
    remainder="drop"
)

cv_folds = min(5, max(2, y_train.value_counts().min()))
cv_strategy = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", list(label_encoder.classes_))
display(data[target_column].value_counts().rename_axis("label").reset_index(name="count"))


In [ ]:
# BLOCK 4: Reusable Utilities
def make_rf_pipeline():
    return Pipeline(
        steps=[
            ("preprocessor", ColumnTransformer(
                transformers=[("num", SimpleImputer(strategy="median"), feature_columns)],
                remainder="drop"
            )),
            (
                "model",
                RandomForestClassifier(
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    class_weight="balanced_subsample"
                )
            )
        ]
    )


def make_xgb_pipeline():
    return Pipeline(
        steps=[
            ("preprocessor", ColumnTransformer(
                transformers=[("num", SimpleImputer(strategy="median"), feature_columns)],
                remainder="drop"
            )),
            (
                "model",
                XGBClassifier(
                    objective="multi:softprob",
                    eval_metric="mlogloss",
                    random_state=RANDOM_STATE,
                    n_estimators=300,
                    learning_rate=0.05,
                    max_depth=5,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_lambda=1.0,
                    min_child_weight=1,
                    tree_method="hist"
                )
            )
        ]
    )


def make_gb_pipeline():
    return Pipeline(
        steps=[
            ("preprocessor", ColumnTransformer(
                transformers=[("num", SimpleImputer(strategy="median"), feature_columns)],
                remainder="drop"
            )),
            ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
        ]
    )


def tune_model(name, pipeline, param_distributions, X_fit, y_fit, n_iter=12):
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="accuracy",
        n_jobs=-1,
        cv=cv_strategy,
        random_state=RANDOM_STATE,
        refit=True,
        verbose=0,
    )
    search.fit(X_fit, y_fit)
    print(f"{name} best CV accuracy: {search.best_score_:.4f}")
    print(f"{name} best params: {search.best_params_}")
    return search.best_estimator_, search.best_score_, search.best_params_


def evaluate_model(model_name, model, X_eval, y_eval, encoder):
    y_pred = model.predict(X_eval)
    acc = accuracy_score(y_eval, y_pred)
    decoded_true = encoder.inverse_transform(y_eval)
    decoded_pred = encoder.inverse_transform(y_pred)
    labels = encoder.classes_

    print(f"\n{model_name} Accuracy: {acc:.4f}")
    print(classification_report(decoded_true, decoded_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(decoded_true, decoded_pred, labels=labels)
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return {"model": model_name, "accuracy": acc, "y_pred": y_pred}


def plot_feature_importance(fitted_pipeline, model_name, top_n=None):
    model = fitted_pipeline.named_steps["model"]
    importances = pd.Series(model.feature_importances_, index=feature_columns).sort_values(ascending=False)
    if top_n is not None:
        importances = importances.head(top_n)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=importances.values, y=importances.index, palette="viridis")
    plt.title(f"{model_name} Feature Importance")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


In [ ]:
# BLOCK 5: Hyperparameter Tuning and Model Training
rf_pipeline = make_rf_pipeline()
xgb_pipeline = make_xgb_pipeline()
gb_pipeline = make_gb_pipeline()

rf_param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 700],
    "model__max_depth": [None, 5, 8, 12, 16, 20],
    "model__min_samples_split": [2, 4, 6, 8],
    "model__min_samples_leaf": [1, 2, 3, 4],
    "model__max_features": ["sqrt", "log2", None]
}

xgb_param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 700],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 2, 3, 5],
    "model__gamma": [0.0, 0.1, 0.2, 0.3],
    "model__reg_lambda": [0.5, 1.0, 1.5, 2.0]
}

gb_param_dist = {
    "model__n_estimators": [100, 150, 200, 250, 300],
    "model__learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "model__max_depth": [2, 3, 4, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__min_samples_split": [2, 4, 6],
    "model__min_samples_leaf": [1, 2, 3]
}

best_rf_model, rf_cv_score, rf_best_params = tune_model("Random Forest", rf_pipeline, rf_param_dist, X_train, y_train)
best_xgb_model, xgb_cv_score, xgb_best_params = tune_model("XGBoost", xgb_pipeline, xgb_param_dist, X_train, y_train)
best_gb_model, gb_cv_score, gb_best_params = tune_model("Gradient Boosting", gb_pipeline, gb_param_dist, X_train, y_train)


In [ ]:
# BLOCK 6: Hybrid Ensemble with StackingClassifier
stacking_estimators = [
    ("random_forest", clone(best_rf_model)),
    ("xgboost", clone(best_xgb_model))
]

stacking_model = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "log_reg",
                LogisticRegression(
                    C=1.0,
                    max_iter=3000,
                    multi_class="auto",
                    random_state=RANDOM_STATE
                )
            )
        ]
    ),
    stack_method="predict_proba",
    passthrough=True,
    cv=cv_strategy,
    n_jobs=-1
)

stacking_model.fit(X_train, y_train)
stacking_cv_score = cross_val_score(
    stacking_model,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=-1
).mean()

print(f"Stacking CV accuracy: {stacking_cv_score:.4f}")


In [ ]:
# BLOCK 7: Evaluation
accuracy_results = {}
cv_results = {
    "Random Forest": rf_cv_score,
    "XGBoost": xgb_cv_score,
    "Gradient Boosting": gb_cv_score,
    "Stacking Ensemble": stacking_cv_score,
}

rf_eval = evaluate_model("Random Forest", best_rf_model, X_test, y_test, label_encoder)
xgb_eval = evaluate_model("XGBoost", best_xgb_model, X_test, y_test, label_encoder)
gb_eval = evaluate_model("Gradient Boosting", best_gb_model, X_test, y_test, label_encoder)
stack_eval = evaluate_model("Stacking Ensemble", stacking_model, X_test, y_test, label_encoder)

accuracy_results[rf_eval["model"]] = rf_eval["accuracy"]
accuracy_results[xgb_eval["model"]] = xgb_eval["accuracy"]
accuracy_results[gb_eval["model"]] = gb_eval["accuracy"]
accuracy_results[stack_eval["model"]] = stack_eval["accuracy"]


In [ ]:
# BLOCK 8: Model Comparison Table
comparison_df = (
    pd.DataFrame(
        {
            "Model": list(accuracy_results.keys()),
            "Test Accuracy": list(accuracy_results.values()),
            "CV Accuracy": [cv_results[name] for name in accuracy_results.keys()],
        }
    )
    .sort_values(by="Test Accuracy", ascending=False)
    .reset_index(drop=True)
)

display(comparison_df.style.format({"Test Accuracy": "{:.4f}", "CV Accuracy": "{:.4f}"}))


In [ ]:
# BLOCK 9: Accuracy Visualization
plt.figure(figsize=(9, 5))
bar_ax = sns.barplot(data=comparison_df, x="Test Accuracy", y="Model", palette="crest")
for container in bar_ax.containers:
    bar_ax.bar_label(container, fmt="%.4f", padding=5)
plt.xlim(0, 1)
plt.title("Model Accuracy Comparison")
plt.xlabel("Test Accuracy")
plt.ylabel("Model")
plt.tight_layout()
plt.show()


In [ ]:
# BLOCK 10: Feature Importance
plot_feature_importance(best_rf_model, "Random Forest")
plot_feature_importance(best_xgb_model, "XGBoost")


In [ ]:
# BLOCK 11: Adaptive Logic and Edge Case Testing
feature_bounds = X_train.agg(["min", "max", "median", "quantile"]).copy()

def build_edge_case_samples(reference_df):
    q10 = reference_df.quantile(0.10)
    q25 = reference_df.quantile(0.25)
    q50 = reference_df.quantile(0.50)
    q75 = reference_df.quantile(0.75)
    q90 = reference_df.quantile(0.90)

    samples = pd.DataFrame(
        [
            {
                "scenario": "High temperature + low moisture",
                "temperature": q90["temperature"],
                "soil_moisture": q10["soil_moisture"],
                "soil_ph": q50["soil_ph"],
                "water_tds": q50["water_tds"],
                "humidity": q25["humidity"],
            },
            {
                "scenario": "High TDS + low moisture",
                "temperature": q75["temperature"],
                "soil_moisture": q10["soil_moisture"],
                "soil_ph": q50["soil_ph"],
                "water_tds": q90["water_tds"],
                "humidity": q50["humidity"],
            },
            {
                "scenario": "Ideal conditions",
                "temperature": q50["temperature"],
                "soil_moisture": q75["soil_moisture"],
                "soil_ph": q50["soil_ph"],
                "water_tds": q25["water_tds"],
                "humidity": q75["humidity"],
            },
        ]
    )
    return samples


def decode_predictions(model, samples, encoder):
    encoded = model.predict(samples[feature_columns])
    return encoder.inverse_transform(encoded)


def test_edge_cases(samples_df, encoder, rf_model, xgb_model, stack_model):
    result_df = samples_df.copy()
    result_df["RF Prediction"] = decode_predictions(rf_model, result_df, encoder)
    result_df["XGB Prediction"] = decode_predictions(xgb_model, result_df, encoder)
    result_df["Stacking Prediction"] = decode_predictions(stack_model, result_df, encoder)
    return result_df


edge_case_samples = build_edge_case_samples(X_train)
edge_case_results = test_edge_cases(edge_case_samples, label_encoder, best_rf_model, best_xgb_model, stacking_model)
display(edge_case_results)


In [ ]:
# BLOCK 12: Final Best Model Summary
best_model_row = comparison_df.iloc[0]
print(f"Best test model: {best_model_row['Model']}")
print(f"Best test accuracy: {best_model_row['Test Accuracy']:.4f}")
print(f"Best CV accuracy: {best_model_row['CV Accuracy']:.4f}")
